<a href="https://colab.research.google.com/github/phuy29032007/Pham-Huy-BT-Ca-Nhan/blob/main/bt_c%C3%A1_nh%C3%A2n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D
from keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt


# 1. TIỀN XỬ LÝ DỮ LIỆU


# Đường dẫn dataset
train_dir = "/content/drive/MyDrive/anhselfie"

# Kích thước ảnh
img_width, img_height = 128, 128

# Batch size
batch_size = 32

# Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode="nearest",
    validation_split=0.2
)


# TẠO TẬP TRAIN


train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset="training"
)

# TẠO TẬP VALIDATION

validation_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(img_width, img_height),
    batch_size=batch_size,
    class_mode="categorical",
    subset="validation"
)

# Số lớp
num_classes = train_generator.num_classes

# 2. XÂY DỰNG MÔ HÌNH CNN

model = Sequential([

    # Layer 1
    Conv2D(
        32,
        (3,3),
        activation="relu",
        input_shape=(img_width, img_height, 3)
    ),
    MaxPooling2D(2,2),

    # Layer 2
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    # Layer 3
    Conv2D(128, (3,3), activation="relu"),
    MaxPooling2D(2,2),

    # Flatten
    Flatten(),

    # Dense Layer
    Dense(128, activation="relu"),

    # Dropout
    Dropout(0.5),

    # Output Layer
    Dense(num_classes, activation="softmax")
])

# 3. BIÊN DỊCH MÔ HÌNH

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# Hiển thị cấu trúc model
model.summary()

# 4. HUẤN LUYỆN MÔ HÌNH
epochs = 30

history = model.fit(
    train_generator,
    epochs=epochs,
    validation_data=validation_generator
)

# 5. VẼ BIỂU ĐỒ ĐỘ CHÍNH XÁC

plt.figure(figsize=(10,5))

plt.plot(
    history.history['accuracy'],
    label="Kết quả huấn luyện"
)

plt.plot(
    history.history['val_accuracy'],
    label="Độ chính xác xác thực"
)

plt.xlabel("Số lần học")
plt.ylabel("Độ chính xác")

plt.title("CNN Image Classification")

plt.legend()

plt.show()

In [ ]:
# Tải và tiền xử lý ảnh kiểm tra
from keras.utils import load_img
import numpy as np

path = "/content/IMG_1779042958637_1779043066067.jpg"
# Tiên đoán loại
img = load_img(path, target_size=(128, 128))

plt.imshow(img)
plt.show()

img = np.array(img)
img = img / 255.0
img = img.reshape(1, 128, 128, 3)

prediction = np.argmax(model.predict(img))

# Ánh xạ loại tới tên người
class_labels = {v: k for k, v in train_generator.class_indices.items()}

person_name = class_labels[prediction]

print(f"Người tiên đoán: {person_name}")